<a href="https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jai-Kulkarni1905/Applied_Search_Intelligence/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# Validation & Research Claim Audit

This section audits whether the project's model evaluation and conclusions
are supported by an appropriate validation design.

The audit covers four questions:

1. Do findings from the FlyRank research paper use labels/outcomes that
   actually support the claims being made?
2. Does our model perform similarly under an honest client-aware split?
3. Does the final feature set contain leakage?
4. Are our conclusions stated as observed, directional, and
   decision-support findings rather than causal claims?

## 1. Two paper findings + my methodology questions

## Finding 1 — Content Lifecycle: Growing vs Declining

### What the paper reports

The FlyRank report separates pages into growing and declining groups and
reports that growing pages are younger on average:

- Growing pages: approximately 185 days old
- Declining pages: approximately 228 days old
- Average word count is almost identical between the groups
  (approximately 1.5K words in both groups).

The report therefore interprets age as the clearer difference between the
two groups rather than content length.

### Where does the label come from?

The label is based on observed movement in the performance window:
pages are divided into "up" and "down" groups according to their observed
traffic trend.

The important methodological point is that this is an observed
performance grouping, not an experimentally assigned treatment.

### Does the validation design carry the claim?

Partially.

The comparison supports an observational statement such as:

> Pages classified as growing in the study were younger on average than
> pages classified as declining.

It does NOT, by itself, establish:

> Making a page younger causes it to grow.

Age is not randomly assigned, and the comparison does not isolate age from
other differences between the page groups.

Therefore the evidence supports an association/directional finding, not a
causal claim.

## Finding 4 — The Freshness Multiplier

### What the paper reports

The report compares growth-to-decline ratios across freshness windows.

The most stable highlighted window is 31–90 days, where the report gives
approximately a 5.43:1 growth-to-decline ratio.

The report also performs a separate comparison among older pages, comparing
pages refreshed recently with pages that had remained stale.

### Where does the label/outcome come from?

The outcomes are observed performance states and changes in the underlying
search/traffic measurements. The comparison is therefore based on what
happened to pages in the observed data.

### Does the validation design carry the claim?

It supports an observational comparison, but not a causal claim.

For example, the data can support:

> In this dataset, recently refreshed pages in the examined cohorts had
> stronger observed performance than comparable stale-page cohorts.

It cannot independently establish:

> Refreshing a page causes its traffic or impressions to increase.

Pages selected for refresh may differ systematically from pages that were
not refreshed. For example, teams may preferentially refresh pages that
already have meaningful visibility or recovery potential.

Therefore the evidence is useful for prioritization and hypothesis
generation, but it is not a causal experiment.

## Overall Research-Paper Audit

The two findings illustrate an important distinction between an observed
pattern and a causal claim.

The paper's comparisons are useful for identifying directional patterns in
content performance. However, because the pages are observationally grouped
rather than randomly assigned to treatments, the findings do not establish
that age or refreshing a page directly causes the observed performance
change.

For this project, the paper is therefore best treated as evidence for
hypothesis generation and contextual grounding, not as causal proof.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## Honest Validation: Client-Grouped Cross-Validation

The model was evaluated using a client-grouped validation design.

This is important because multiple pages can belong to the same client. A
random row-level split could therefore place pages from the same client in
both training and test sets, making the evaluation artificially easy.

Instead, the validation keeps clients separated between training and testing.
The model is therefore evaluated on clients that were not used to train that
fold.

Five validation folds were evaluated.

### Fold-Level Precision@50 Results

| Fold | Test Clients | Test Rows | Base Rate | Baseline Precision@50 | RF Precision@50 | RF Lift vs Baseline | RF Lift vs Base Rate |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 0 | 6 | 28,294 | 0.513 | 0.640 | 0.780 | +0.140 | +0.267 |
| 1 | 9 | 28,295 | 0.538 | 0.460 | 0.640 | +0.180 | +0.102 |
| 2 | 8 | 28,295 | 0.350 | 0.740 | 0.680 | -0.060 | +0.330 |
| 3 | 10 | 28,291 | 0.517 | 0.900 | 0.840 | -0.060 | +0.323 |
| 4 | 10 | 28,292 | 0.396 | 0.820 | 0.620 | -0.200 | +0.224 |

### What the Fold Results Show

The Random Forest does not outperform the baseline in every fold.

It improves on the baseline in folds 0 and 1, while the baseline performs
better in folds 2, 3, and 4.

However, the Random Forest remains above the corresponding base rate in all
five folds.

This distinction is important: the model provides evidence of useful ranking
signal relative to the underlying prevalence, but the results do not show
that the learned model consistently dominates the hand-built baseline for
every client group.


## Overall Model Performance

The model produced the following training and test results:

| Metric | Train | Test |
|---|---:|---:|
| ROC-AUC | 0.6897 | 0.6515 |
| Average Precision | 0.6259 | 0.6033 |

The decrease from training to test performance indicates some expected
generalization gap.

The test ROC-AUC of 0.6515 and test Average Precision of 0.6033 are the
appropriate figures for judging performance on held-out data.

## Before vs Honest Validation

The final validation design uses client-grouped folds rather than treating
individual page rows as independent observations.

The earlier evaluation and the final honest evaluation should not be compared
as if they were measuring exactly the same experimental condition unless the
earlier split configuration is explicitly recorded.

For the final report, the client-grouped test results are treated as the
authoritative validation results because they better reflect the project's
data structure.

### Final honest-validation results

- Test ROC-AUC: **0.6515**
- Test Average Precision: **0.6033**
- Random Forest Precision@50: **0.780, 0.640, 0.680, 0.840, 0.620** across folds
- Random Forest exceeded the base rate in **all five folds**
- Random Forest exceeded the baseline in **2 of 5 folds**

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

# Leakage Audit

The final feature set was checked for information that would not have been
available at the decision point.

The audit focuses on:

1. Target leakage
2. Future-information leakage
3. Product-decision leakage
4. Target-derived features
5. Client overlap between training and test data
6. Duplicate observations

## Leakage Test

A leakage audit was already performed in `w05_model.ipynb` **before training any model**. The audit checked the feature construction and data pipeline to ensure that target information, future information, and other decision-derived information were not available to the model as inputs.

Because this check was completed before model training, the leakage audit was not performed after observing model performance and therefore did not depend on the model's results.

The same validated feature set is carried forward into this notebook.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

### Original / overly strong claim

> The model identifies the pages that should be refreshed and predicts which
> pages will recover.

### Safe claim

> On the evaluated client-grouped validation folds, the Random Forest showed
> useful ranking signal relative to the underlying base rate, with test
> ROC-AUC of 0.6515 and test Average Precision of 0.6033. Its Precision@50
> exceeded the base rate in all five folds, although it did not outperform the
> transparent baseline in every fold. These results suggest that combining
> observable search and content signals can support prioritization of pages
> for human review. The model provides directional decision-support evidence;
> it does not establish that refreshing a selected page will cause recovery.

## What the Evidence Supports

### We can say

- The model was evaluated using client-grouped validation.
- The held-out test ROC-AUC was **0.6515**.
- The held-out Average Precision was **0.6033**.
- Random Forest Precision@50 was above the base rate in all five validation
  folds.
- Random Forest beat the transparent baseline in 2 of the 5 folds.
- The model provides useful directional evidence for ranking/prioritization
  under the evaluated conditions.
- The output can support a human content-review queue.

### We cannot say

- The model guarantees which pages will recover.
- Refreshing a selected page causes recovery.
- The model predicts Google's ranking algorithm.
- The model will perform identically on every future client.
- The learned model is universally better than the baseline.
- The observational FlyRank findings prove a causal effect of freshness.

# Final Validation Verdict

## Verdict: Suitable for decision-support, with explicit limitations

The final model was evaluated using client-grouped validation so that test
clients were kept separate from training clients.

The Random Forest achieved:

- **Test ROC-AUC:** 0.6515
- **Test Average Precision:** 0.6033

Across the five client-grouped folds, Random Forest Precision@50 was:

- 0.780
- 0.640
- 0.680
- 0.840
- 0.620

The model exceeded the base rate in every fold, but it exceeded the
transparent baseline in only two of the five folds.

Therefore, the evidence does not support claiming that the learned model
universally outperforms the baseline. Instead, it supports the narrower
conclusion that the model contains useful ranking signal that can assist
content-review prioritization under the evaluated conditions.

The system should be treated as a decision-support tool. A high-ranked page
is a candidate for human review, not a guaranteed recovery opportunity.

Likewise, the observational research findings used to motivate the project
support directional associations but do not establish that refreshing a page
causes improved search performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.